In [ ]:
"""
# BigAlpha 2026 —— PatchTST-Lite 第六轮提交入口

实验：r6_09_d60_lowreg_bridge

同目录文件：
- train_patchtst_lite_best_submit_r6_09_d60_lowreg_bridge.py
- train_patchtst_lite_best_submit_r6_09_d60_lowreg_bridge.ipynb
- train_patchtst_lite_best_submit_r6_09_d60_lowreg_bridge.json

模型 JSON 不存在时，Python 模块会自动训练并将权重保存到本目录。
公榜正式提交前，应将真实训练权重与 py/ipynb 一并放入本套件。
"""

import os
import sys

try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _HERE = os.getcwd()

if _HERE not in sys.path:
    sys.path.insert(0, _HERE)

from train_patchtst_lite_best_submit_r6_09_d60_lowreg_bridge import (
    CFG,
    FEATURE_COLS,
    SELECTED_EXPERIMENT_NAME,
    SUBMISSION_MODEL_FILENAME,
    SUBMIT_HYPERPARAMS,
    train_and_save,
    main as _model_main,
)

MODEL_PATH = os.path.join(_HERE, SUBMISSION_MODEL_FILENAME)
CFG.model_path = MODEL_PATH


def main(datasources, start_date, end_date):
    CFG.model_path = MODEL_PATH
    score_data = _model_main(datasources, start_date, end_date)

    expected_columns = ["date", "instrument", "score"]
    if list(score_data.columns) != expected_columns:
        raise RuntimeError(
            f"提交结果列错误：{list(score_data.columns)}；"
            f"应为：{expected_columns}"
        )

    return score_data


if __name__ == "__main__":
    datasources = {"bar30m": "bigalpha_2026_stock_bar30m"}
    start_date = CFG.public_start
    end_date = CFG.public_end

    print(f"selected experiment: {SELECTED_EXPERIMENT_NAME}")
    print(f"feature count      : {len(FEATURE_COLS)}")
    print(f"model path         : {MODEL_PATH}")
    print(f"hyperparameters    : {SUBMIT_HYPERPARAMS}")

    if not os.path.exists(MODEL_PATH):
        print(f"未发现模型权重 {MODEL_PATH}，开始训练...")
        train_and_save(
            data_source_name=CFG.data_source,
            train_start=CFG.train_start,
            train_end=CFG.train_end,
            cfg=CFG,
        )

    score_data = main(datasources, start_date, end_date)
    print(score_data.head())
    print(score_data.shape)

    try:
        from bigmodule import M
        result = M.bigalpha_eval._latest(
            factor_data=score_data,
            show=True,
        )
    except ImportError:
        print("未检测到 bigmodule，跳过本地画图评估。")
    except Exception as exc:
        print(f"评估模块执行失败：{repr(exc)}")
